# 32 — Multi-NR LGBM: Morgan+RDKit + ProtBERT Protein Embeddings

Same multi-NR structure as nb30 but uses **ProtBERT** (1024-dim) protein embeddings
instead of ESM-2 (320-dim). ProtBERT is trained on BFD/UniRef and captures
different biochemical properties than ESM-2.

- Compound features: Morgan (2048) + RDKit (217) = 2265-dim
- Protein features: ProtBERT global embedding (1024-dim)
- Total input: **3289 features**

In [1]:
import sys, warnings, json
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.featurize import combined, impute
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5

LGBM_PARAMS = dict(
    n_estimators=1200, num_leaves=64, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.2,
    min_child_samples=10, n_jobs=4, verbose=-1
)

TARGET_WEIGHTS = {
    'PXR':   1.0,
    'VDR':   0.50,
    'FXR':   0.30,
    'LXRa':  0.25,
    'RXRa':  0.25,
    'PPARg': 0.15,
    'PPARa': 0.15,
}

print('Setup complete.')

Setup complete.


## 1. Load data and ProtBERT protein embeddings

In [2]:
train = load_train()
te    = load_test()
nr    = pd.read_parquet('../data/external/chembl_nr_targets.parquet')

# Load ProtBERT protein embeddings
protbert_arr = np.load(DATA_PROCESSED / 'nr_protbert_embeddings.npy')  # (n_proteins, 1024)
with open(DATA_PROCESSED / 'nr_protbert_names.json') as f:
    protbert_names = json.load(f)
protbert_lookup = {name: protbert_arr[i] for i, name in enumerate(protbert_names)}
PROTEIN_DIM = protbert_arr.shape[1]  # 1024

print(f'CRC train: {len(train):,}  |  ChEMBL NR: {len(nr):,}  |  Test: {len(te):,}')
print(f'ProtBERT embedding dim: {PROTEIN_DIM}')
print(f'Available targets: {protbert_names}')
print(nr['target_name'].value_counts())

CRC train: 4,139  |  ChEMBL NR: 11,511  |  Test: 513
ProtBERT embedding dim: 1024
Available targets: ['PXR', 'VDR', 'FXR', 'LXRa', 'RXRa', 'PPARg', 'PPARa']
target_name
PPARg    4311
FXR      3187
RXRa     1364
LXRa     1175
PXR       947
VDR       523
PPARa       4
Name: count, dtype: int64


## 2. Featurize compounds (Morgan + RDKit)

In [3]:
# Filter ChEMBL NR
nr_filt = nr[(nr['pec50'] >= 3.0) & (nr['pec50'] <= 10.0)].copy()
nr_filt = nr_filt[nr_filt['target_name'].isin(protbert_lookup)].copy()
nr_filt['weight'] = nr_filt['target_name'].map(TARGET_WEIGHTS).fillna(0.1)
print(f'ChEMBL NR after quality filter: {len(nr_filt):,}')

print('Featurizing CRC train...')
X_tr_chem = impute(combined(train['smiles'].tolist()))   # (4139, 2265)
y_tr = train['pec50'].values

print('Featurizing ChEMBL NR...')
X_nr_chem_raw = combined(nr_filt['smiles'].tolist())
valid_nr_mask = ~np.all(np.isnan(X_nr_chem_raw), axis=1)
X_nr_chem_raw = X_nr_chem_raw[valid_nr_mask]
nr_filt_valid = nr_filt.iloc[valid_nr_mask].reset_index(drop=True)
X_nr_chem = impute(X_nr_chem_raw)
print(f'  Valid ChEMBL NR compounds: {len(nr_filt_valid):,} / {len(nr_filt):,}')

print('Featurizing test...')
X_te_chem = impute(combined(te['smiles'].tolist()))     # (513, 2265)

CHEM_DIM = X_tr_chem.shape[1]  # 2265
TOTAL_DIM = CHEM_DIM + PROTEIN_DIM  # 3289
print(f'\nChem: {CHEM_DIM}  |  Protein: {PROTEIN_DIM}  |  Total: {TOTAL_DIM}')

ChEMBL NR after quality filter: 11,496
Featurizing CRC train...


Featurizing ChEMBL NR...


  Valid ChEMBL NR compounds: 11,496 / 11,496
Featurizing test...



Chem: 2265  |  Protein: 1024  |  Total: 3289


## 3. Build multi-NR feature matrices

In [4]:
pxr_emb = protbert_lookup['PXR']  # (1024,)

# CRC train: [Morgan+RDKit | PXR ProtBERT embedding]
pxr_tile_tr = np.tile(pxr_emb, (len(X_tr_chem), 1))   # (4139, 1024)
X_tr_full = np.hstack([X_tr_chem, pxr_tile_tr])         # (4139, 3289)
w_tr = np.ones(len(y_tr))

# ChEMBL NR: [Morgan+RDKit | per-target ProtBERT embedding]
nr_protein_embs = np.stack(
    [protbert_lookup[t] for t in nr_filt_valid['target_name']], axis=0
)  # (n_nr, 1024)
X_nr_full = np.hstack([X_nr_chem, nr_protein_embs])     # (n_nr, 3289)
y_nr = nr_filt_valid['pec50'].values
w_nr = nr_filt_valid['weight'].values

# Test: [Morgan+RDKit | PXR ProtBERT embedding]
pxr_tile_te = np.tile(pxr_emb, (len(X_te_chem), 1))   # (513, 1024)
X_te_full = np.hstack([X_te_chem, pxr_tile_te])         # (513, 3289)

print(f'Train (CRC): {X_tr_full.shape}')
print(f'NR augment:  {X_nr_full.shape}')
print(f'Test:        {X_te_full.shape}')

Train (CRC): (4139, 3289)
NR augment:  (11496, 3289)
Test:        (513, 3289)


## 4. Scaffold 5-fold CV on CRC train

In [5]:
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    X_fold = np.vstack([X_tr_full[tr_idx], X_nr_full])
    y_fold = np.concatenate([y_tr[tr_idx], y_nr])
    w_fold = np.concatenate([w_tr[tr_idx], w_nr])

    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_fold, y_fold, sample_weight=w_fold)

    oof[va_idx] = m.predict(X_tr_full[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    fold_metrics.append(met)
    print(f'  Fold {fold_i+1}: RAE={met["RAE"]:.4f}  Spearman={met["Spearman"]:.4f}')

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f'\nOOF RAE (global): {oof_rae:.4f}')
print(f'Mean fold RAE:    {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'Mean Spearman:    {cv_df["Spearman"].mean():.4f}')
print(f'\nComparison:')
print(f'  Morgan+ESM-2 (nb30):     see oof_morgan_esm2_nr.npy')
print(f'  Morgan+ProtBERT (this):  {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_morgan_protbert_nr.npy', oof)

  Fold 1: RAE=0.5152  Spearman=0.7711


  Fold 2: RAE=0.5973  Spearman=0.6837


  Fold 3: RAE=0.6136  Spearman=0.6737


  Fold 4: RAE=0.5666  Spearman=0.7021


  Fold 5: RAE=0.6054  Spearman=0.6903

OOF RAE (global): 0.5749
Mean fold RAE:    0.5796 +/- 0.0401
Mean Spearman:    0.7042

Comparison:
  Morgan+ESM-2 (nb30):     see oof_morgan_esm2_nr.npy
  Morgan+ProtBERT (this):  0.5749


## 5. Full retrain + test predictions + submission

In [6]:
X_full = np.vstack([X_tr_full, X_nr_full])
y_full = np.concatenate([y_tr, y_nr])
w_full = np.concatenate([w_tr, w_nr])

final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_full, y_full, sample_weight=w_full)

te_preds = np.clip(final_m.predict(X_te_full), y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_morgan_protbert_nr.npy', te_preds)

sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'

out_path = SUBMISSIONS / '32_protbert_multinr.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'OOF RAE: {oof_rae:.4f}')
print(f'Test pEC50: min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}')
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\32_protbert_multinr.csv
OOF RAE: 0.5749
Test pEC50: min=2.25  median=4.95  max=6.16
count    513.000
mean       4.819
std        0.647
min        2.249
25%        4.454
50%        4.954
75%        5.289
max        6.160
Name: pEC50, dtype: float64
